In [ ]:
class CondConv2D(nn.Module):
    """
    Conditionally Parameterized Convolutions as Branch Attention Mechanism.
    Multiple convolutional experts are combined dynamically based on input.
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, num_experts=3):
        super(CondConv2D, self).__init__()
        self.num_experts = num_experts

        # Define multiple convolution experts (branches)
        self.expert_convs = nn.ModuleList([
            nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding)
            for _ in range(num_experts)
        ])

        # Gating mechanism to decide which expert should be applied more
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  # Global context (pooling over spatial dimensions)
            nn.Flatten(),
            nn.Linear(in_channels, num_experts),  # Linear layer to output attention for each expert
            nn.Softmax(dim=1)  # Softmax to get probabilities (attention weights)
        )

    def forward(self, x):
        batch_size, _, _, _ = x.size()

        # Compute attention weights (gates) for each expert (branch)
        gates = self.gate(x)  # Shape: [batch_size, num_experts]

        # Apply each expert's convolution and compute weighted sum of the results
        out = sum(gates[:, i].view(batch_size, 1, 1, 1) * self.expert_convs[i](x)
                  for i in range(self.num_experts))

        return out